# Question 20 - 14. Longest Common Prefix

Write a function to find the longest common prefix string amongst an array of strings.

If there is no common prefix, return an empty string `""`.

**Example 1:**

    Input: strs = ["flower","flow","flight"]
    Output: "fl"

**Example 2:**

    Input: strs = ["dog","racecar","car"]
    Output: ""
    Explanation: There is no common prefix among the input strings.

**Constraints:**

- `1 <= strs.length <= 200`
- `0 <= strs[i].length <= 200`
- `strs[i]` consists of only lowercase English letters if it is non-empty.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Vertical scanning (compare column by column, stop at first mismatch)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (try every prefix length) | O(m · S) | O(m) |
| 2️⃣ Optimized (vertical scan) | O(S) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"I want the longest string that every word starts with. The answer can never be longer than the **shortest** word, and it's a prefix of the first word, so I can build it from `strs[0]`."*

- Can a string be empty? → **Yes** (length 0). Then the answer is `""`.
- Only one string? → The answer is that string.
- Lowercase only, so no case folding needed.

**Notation:** `n` = number of strings, `m` = length of the shortest string, `S` = total characters across all strings.

### Step 1 · Brute force

**Idea:** Try every prefix of `strs[0]` from longest to shortest. Return the first one that every string `startswith`.

🗣️ *"The naive approach tries candidate prefixes from longest to shortest and checks each against all strings. In the worst case that's about m candidates, each checked against all n strings in O(m), so O(n·m²). That's wasteful: once I know the first k characters match everywhere, I shouldn't recheck them for length k+1."*

In [1]:
class SolutionBrute:
    def longestCommonPrefix(self, strs: list[str]) -> str:
        first = strs[0]
        for length in range(len(first), -1, -1):
            prefix = first[:length]
            if all(s.startswith(prefix) for s in strs):
                return prefix
        return ""

### Step 2 · Optimize

**Bottleneck:** brute force re-verifies the same leading characters for every candidate length.

**Key insight: check one column at a time.** Look at character `i` of the first word, and verify that *every* string has the same character at position `i`. Stop at the first column where any string is too short or mismatches. Each character is compared **once**.

🗣️ *"I'll scan vertically. For each index i in the first string, check that every other string has the same character at i. The moment one is too short or differs, the prefix ends at i. Each character is compared at most once, so it's O(S) time, and O(1) extra space because I just return a slice."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Try every prefix length | ❌ | Re-checks the same characters repeatedly. |
| Horizontal: `prefix = strs[0]`, shrink it against each string | ✅ | O(S) too, but it can do extra work when the first string is long and a later one is short. |
| **Vertical scan (column by column)** | ✅ (best) | Stops at the **first** mismatch across all strings, so it never reads beyond the answer + 1 column. |
| Sort, then compare only first and last | ⚠️ | Elegant (the most different strings end up at the extremes), but sorting costs O(S log n). Using `min()`/`max()` instead of sorting gives O(S). |
| Trie of all strings, walk while a node has one child | ❌ | O(S) time **and O(S) space**. Overkill for a single query. Only worth it for many prefix queries on the same set. |
| Binary search on prefix length | ⚠️ | O(S log m). No better than vertical. |

In [2]:
class Solution:
    def longestCommonPrefix(self, strs: list[str]) -> str:
        first = strs[0]
        for i, ch in enumerate(first):
            for s in strs[1:]:
                if i == len(s) or s[i] != ch:   # too short or mismatch
                    return first[:i]
        return first

### Step 3 · Toy example walkthrough

`strs = ["flower", "flow", "flight"]`

| column i | first[i] | "flow"[i] | "flight"[i] | all equal? |
|---|---|---|---|---|
| 0 | f | f | f | ✅ |
| 1 | l | l | l | ✅ |
| 2 | o | o | **i** | ❌ → return `first[:2]` = **"fl"** |

We compared 2 full columns plus part of the third. Nothing after the mismatch was read.

Example 2: `["dog", "racecar", "car"]` → column 0: `d` vs `r` ❌ → `""` immediately.

In [3]:
cases = [
    (["flower", "flow", "flight"], "fl"),
    (["dog", "racecar", "car"], ""),
    (["alone"], "alone"),
    (["", "abc"], ""),
    (["abc", ""], ""),
    (["ab", "a"], "a"),
    (["same", "same", "same"], "same"),
    (["interview", "internet", "interval"], "inter"),
]
for strs, expected in cases:
    assert SolutionBrute().longestCommonPrefix(strs) == expected
    assert Solution().longestCommonPrefix(strs) == expected
    print(strs, "->", repr(expected))
print("All tests passed ✅")

['flower', 'flow', 'flight'] -> 'fl'
['dog', 'racecar', 'car'] -> ''
['alone'] -> 'alone'
['', 'abc'] -> ''
['abc', ''] -> ''
['ab', 'a'] -> 'a'
['same', 'same', 'same'] -> 'same'
['interview', 'internet', 'interval'] -> 'inter'
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Try every prefix | O(n · m²) | O(m) | Rechecks shared characters per candidate |
| Sort + compare ends | O(S log n) | O(m) | Sorting dominates |
| Trie | O(S) | O(S) | Builds the full structure |
| **Vertical scan** | **O(S)** (really O(n · answer_len)) | **O(1)** | Each character is compared at most once, and we stop at the first bad column |

**Complexity, explained:** the outer loop runs at most `len(answer) + 1` times. Each run checks all `n` strings once. So the work is O(n · (len(answer)+1)) ≤ O(S). The only allocation is the returned slice (output), so extra space is O(1).

**Edge cases:** a single string, an empty string anywhere, the first string longer than the others (`["ab","a"]`), all identical.

**Follow-up:** *Many queries ("LCP of this new word with the set")?* → A trie becomes worth its O(S) build cost.

🗣️ **30-second recap:** *"I scan column by column using the first string as the reference. For each position, I check that every string is long enough and has the same character there. At the first failure I return the prefix so far. Each character is checked at most once, so it's O(S) time and O(1) extra space."*